# Benchmark lab

Try configs and models interactively, then promote the good ones to a YAML benchmark.

This uses the project's own code — `ml/data.py` (features), `ml/splits.py`, `ml/models.py`,
`ml/metrics.py`, `ml/benchmark.py` — so a result here matches `pixi run benchmark`.
The only difference: nothing is logged to MLflow until you save a config and run it from
the terminal (last section).

**Before you start:** kernel = `.pixi/envs/default/bin/python`, and `pixi run build` has
run. The data is loaded once (cell 2); re-run only the cells below it while experimenting.

In [ ]:
import os
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)                 # the warehouse's staging views read raw files by paths relative to the repo root
sys.path.insert(0, str(ROOT))

import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml
from sklearn.calibration import calibration_curve
from sklearn.metrics import precision_recall_curve

from ml import benchmark, data, models, splits
from ml.metrics import score

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

## 1. Load the data (once — takes ~20 s)

One row per region per 6-hour block, every feature, years with weather. The label `y`
and the onset flag are added per threshold later, so changing the threshold doesn't reload.

In [ ]:
YEARS = [2024, 2025]          # years with HRRR weather downloaded (add 2020-2023 once fetched)
BASE = data.load(None, YEARS)
print(f"{len(BASE):,} region-blocks, {BASE.region_fips.nunique()} regions, {BASE.block_start.min():%Y-%m-%d} → {BASE.block_start.max():%Y-%m-%d}")

## 2. What you can combine

Feature groups (from `ml/data.py`) — combine with `+`, remove with `-`, e.g.
`"weather+place-canopy"`. Models (from `ml/models.py`) and their hyperparameters go in
`params`; a list means "try each value".

In [ ]:
for name, cols in data.FEATURE_GROUPS.items():
    print(f"{name:8s} ({len(cols):2d}): {', '.join(cols)}")
print("\nmodels:", ", ".join(sorted(models.REGISTRY)))

## 3. Your experiment

Same shape as a YAML config in `ml/benchmarks/`. Keep it small while exploring — a
`county` split fits 5 models per run, so it's ~5x slower than `time`.

In [ ]:
CONFIG = {
    "name": "lab",
    "data": {"threshold_pct": [5], "years": YEARS, "train_on": ["all"]},
    "rank_by": "onset_pr_auc",
    "splits": [
        {"kind": "time", "train_years": [2024], "test_years": [2025]},
        # {"kind": "county", "folds": 5},
    ],
    "features": ["weather", "weather+place", "weather+place-canopy"],
    "models": [
        {"name": "climatology"},
        {"name": "xgboost", "params": {"n_estimators": 400, "learning_rate": 0.03, "max_depth": [4, 6],
                                       "min_child_weight": 5, "subsample": 0.8, "colsample_bytree": 0.8}},
    ],
}
runs = benchmark.plan(CONFIG)
print(f"{len(runs)} runs")
for r in runs:
    print(f"  >={r['threshold_pct']}% {r['train_on']:6s} {benchmark.split_label(r['split'])[:30]:30s} "
          f"{r['model']:12s} {r['features']:28s} {r['params']}")

## 4. Run it

In [ ]:
labeled = {t: data.label(BASE, t) for t in {r["threshold_pct"] for r in runs}}
rows = []
for i, run in enumerate(runs, 1):
    scores = benchmark.evaluate(labeled[run["threshold_pct"]], run)
    rows.append({"threshold": run["threshold_pct"], "train_on": run["train_on"],
                 "split": run["split"]["kind"], "model": run["model"], "features": run["features"],
                 "params": str(run["params"]), **scores})
    print(f"[{i}/{len(runs)}] {run['model']:12s} {run['features']:28s} onset PR-AUC {scores['onset_pr_auc']:.3f} "
          f"({scores['onset_pr_auc_lift']:.1f}x)  all {scores['pr_auc']:.3f}  {scores['fit_seconds']:.0f}s")

RESULTS = pd.DataFrame(rows)
show = ["threshold", "train_on", "split", "model", "features", "params", "onset_pr_auc", "onset_pr_auc_lift",
        "onset_precision_top1", "onset_recall_top1", "pr_auc", "roc_auc", "brier", "fit_seconds"]
RESULTS[show].sort_values(["threshold", "split", "onset_pr_auc"], ascending=[True, True, False]).round(4)

## 5. Compare

PR-AUC on new outages for every run. The dotted line is no skill (the onset event rate).

In [ ]:
for (thr, split), g in RESULTS.groupby(["threshold", "split"]):
    g = g.sort_values("onset_pr_auc")
    labels = g.model + " · " + g.features + np.where(g.params != "{}", " · " + g.params, "")
    fig, ax = plt.subplots(figsize=(9, 0.4 * len(g) + 1))
    ax.barh(labels, g.onset_pr_auc, color=np.where(g.model.isin(benchmark.BASELINES), "#898781", "#2a78d6"))
    ax.axvline(g.onset_event_rate.iloc[0], ls=":", c="#898781")
    ax.set_xlabel("onset PR-AUC")
    ax.set_title(f">= {thr}% out · {split} split")
    ax.spines[["top", "right"]].set_visible(False)
    plt.tight_layout()
    plt.show()

## 6. Look inside one model

Trains one model on the first year and predicts the last, then shows:
- the **precision–recall curve**, all blocks vs. new outages — where the gap comes from
- **calibration** — when it says 10%, does it happen ~10% of the time?
- which features it **uses** (built-in importance; see `export_app_data.py` for permutation importance)

In [ ]:
INSPECT = {"model": "xgboost", "features": "weather+place+history+recent", "threshold_pct": 5,
           "params": {"n_estimators": 400, "learning_rate": 0.03, "max_depth": 6, "min_child_weight": 5,
                      "subsample": 0.8, "colsample_bytree": 0.8}}

df = data.label(BASE, INSPECT["threshold_pct"])
cols = data.resolve_features(INSPECT["features"])
[(_, tr, te)] = list(splits.make_splits(df, "time", train_years=[min(YEARS)], test_years=[max(YEARS)]))
start = time.time()
model = models.build(INSPECT["model"], INSPECT["params"]).fit(df.iloc[tr][cols], df.iloc[tr]["y"])
test = df.iloc[te].assign(prob=model.predict_proba(df.iloc[te][cols])[:, 1])
print(f"trained in {time.time() - start:.0f}s")
pd.DataFrame({"all blocks": score(test.y, test.prob), "new outages": score(test.y[test.is_onset], test.prob[test.is_onset])}).round(4)

In [ ]:
fig, (a, b) = plt.subplots(1, 2, figsize=(12, 4.5))
for name, sub, color in [("all blocks", test, "#2a78d6"), ("new outages", test[test.is_onset], "#eb6834")]:
    p, r, _ = precision_recall_curve(sub.y, sub.prob)
    a.plot(r, p, color=color, lw=2, label=f"{name} (event rate {sub.y.mean():.2%})")
    frac, mean_pred = calibration_curve(sub.y, sub.prob, n_bins=10, strategy="quantile")
    b.plot(mean_pred, frac, "o-", color=color, lw=2, label=name)
a.set(xlabel="recall", ylabel="precision", title="Precision–recall")
lim = test.prob.quantile(0.999)
b.plot([0, lim], [0, lim], ":", c="#898781", label="perfect calibration")
b.set(xlabel="predicted probability", ylabel="observed frequency", title="Calibration (deciles of risk)")
for ax in (a, b):
    ax.legend(frameon=False)
    ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
if hasattr(model, "feature_importances_"):
    imp = pd.Series(model.feature_importances_, index=cols).sort_values().tail(15)
    ax = imp.plot.barh(figsize=(7, 5), color="#2a78d6", title="Built-in feature importance (top 15)")
    ax.spines[["top", "right"]].set_visible(False)
    plt.tight_layout()
    plt.show()

### Where does it miss?

New outages the model ranked low — read their weather and place. Misses that share a
pattern are your next feature idea.

In [ ]:
misses = test[test.is_onset & (test.y == 1)].sort_values("prob").head(15)
misses[["region", "block_start", "peak_pct_out", "prob", "gust_max_ms", "cape_max_jkg", "precip_sum_mm",
        "storm_reports", "canopy_mean_pct", "customers"]].round(3)

## 7. Promote it to a real benchmark

Saves `CONFIG` as YAML. Then run it from the terminal — that logs every run to MLflow and
writes `ml/results/benchmark_<name>.csv` + a leaderboard:

```bash
pixi run benchmark ml/benchmarks/<name>.yaml
```

In [ ]:
path = ROOT / "ml" / "benchmarks" / f"{CONFIG['name']}.yaml"
path.write_text(yaml.safe_dump(CONFIG, sort_keys=False))
print(f"saved {path.relative_to(ROOT)} — run: pixi run benchmark {path.relative_to(ROOT)}")